# EXP_010 — Text-Only Baseline (XLM-R + MSE)
**Phase 1 | Baseline Establishment**
Research question: How strong is review text alone for predicting the five targets?
- Text model: `xlm-roberta-base` | Image branch: DISABLED | Fusion: none | Loss: MSE | Seed: 42 | AMP: enabled

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13272, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 13272 (delta 97), reused 106 (delta 63), pack-reused 13129 (from 1)
Receiving objects: 100% (13272/13272), 873.17 MiB | 31.03 MiB/s, done.
Resolving deltas: 100% (335/335), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!gdown --id 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD
From (redirected): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD&confirm=t&uuid=21388ac4-29ea-4a95-9412-6370b1356e4c
To: /content/SE365/data.zip
100% 4.02G/4.02G [00:48<00:00, 83.4MB/s]
total 1344
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 23 02:26 ..
drwxr-xr-x  2 root root 1359872 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_010_text_only_xlmr_mse'
DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')

Artifacts will be saved to: /content/drive/MyDrive/SE365/experiments/EXP_010_text_only_xlmr_mse


### STEP 5: Train

In [ ]:
!python main.py \
  --mode train_text \
  --text_model_name xlm-roberta-base \
  --epochs 20 \
  --batch_size 32 \
  --lr 2e-5 \
  --grad_accum_steps 1 \
  --patience 5 \
  --loss_fn mse \
  --seed 42 \
  --use_amp \
  --exp_id EXP_010_text_only_xlmr_mse \
  --exp_dir ./experiments

====== MODE: TRAIN_TEXT ======
Using device: cuda
Seed: 42 | Experiment: EXP_010_text_only_xlmr_mse
config.json: 100% 615/615 [00:00<00:00, 1.56MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 108kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 14.2MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 17.5MB/s]
preprocessor_config.json: 100% 368/368 [00:00<00:00, 1.61MB/s]
model.safetensors: 100% 1.12G/1.12G [00:02<00:00, 399MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6466.17it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

### STEP 6: Save to Drive + print metrics

In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

Saved to /content/drive/MyDrive/SE365/experiments/EXP_010_text_only_xlmr_mse

=== EXP_010_text_only_xlmr_mse Results ===
Loss (val)   : 2.8301

             MAE      RMSE      R2
  food     : 1.2717   1.7472   0.4200
  price    : 1.2928   1.7583   0.3078
  atmos    : 1.2510   1.6438   0.3037
  service  : 1.3135   1.7583   0.3970
  overall  : 1.0880   1.4800   0.4620

  mean_mae   : 1.2434
  aspect_mae : 1.2823
  overall_mae: 1.0880
